# Customer Churn Prediction and Retention Analysis

End-to-end telecom analytics and machine learning case study using the Kaggle Telco Customer Churn dataset.

Dataset source: https://www.kaggle.com/datasets/blastchar/telco-customer-churn

## 1. Business Problem Understanding

### Problem statement

A telecom company wants to predict whether a customer is likely to discontinue service and understand which customer, service, billing, and contract factors contribute to churn.

### Business impact

- Customer acquisition is expensive, so replacing churned customers reduces margin.
- Churn creates recurring revenue loss and lowers customer lifetime value.
- Retention campaigns have a cost, so outreach should be prioritized by risk and value.
- Better churn monitoring can help support, pricing, and product teams fix root causes.

### Business goals

- Identify high-risk churn customers before cancellation.
- Explain the strongest churn drivers.
- Recommend retention strategies by customer segment.
- Build a reproducible workflow for dashboarding and future monitoring.

### KPIs

- Churn rate
- Recall for churn customers
- ROC-AUC
- Monthly recurring revenue at risk
- High-risk customer count
- Churn rate by contract, tenure, payment method, and service bundle

## 2. Setup and Data Loading

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.append(str(PROJECT_ROOT / 'src'))

from IPython.display import Image, display

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from churn_data import (
    FEATURE_CATEGORIES,
    TARGET_COLUMN,
    build_preprocessor,
    clean_telco_data,
    data_quality_report,
    dataset_overview,
    engineer_features,
    load_telco_data,
    make_train_test_split,
    save_processed_data,
)
from churn_modeling import (
    extract_feature_importance,
    plot_best_confusion_matrix,
    plot_feature_importance,
    plot_roc_curves,
    save_model_artifacts,
    select_best_model,
    train_and_evaluate_models,
)
from churn_visuals import churn_rate_by
from paths import RAW_DATA_PATH

sns.set_theme(style='whitegrid', palette='Set2')
pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', '{:,.3f}'.format)

In [ ]:
raw_df = load_telco_data(RAW_DATA_PATH)
raw_df.head()

## 3. Dataset Overview and Data Quality Assessment

In [ ]:
overview = dataset_overview(raw_df)
overview['shape'], overview['columns']

In [ ]:
raw_df.info()

In [ ]:
raw_df.describe(include='all').T

In [ ]:
data_quality_report(raw_df)

### Feature categories

- Demographic: gender, senior citizen status, partner, dependents.
- Service-related: phone, internet, security, backup, protection, support, streaming services.
- Account-related: tenure, contract, billing, payment method, monthly charges, total charges.
- Target: Churn, where `Yes` means the customer discontinued service.

In [ ]:
pd.DataFrame([(category, ', '.join(columns)) for category, columns in FEATURE_CATEGORIES.items()], columns=['category', 'features'])

## 4. Data Cleaning and Preprocessing

Cleaning steps:

- Strip whitespace from column names and string values.
- Remove duplicate customer records.
- Convert `TotalCharges` to numeric.
- Repair blank `TotalCharges` values for zero-tenure customers.
- Remove invalid tenure, monthly charge, and total charge records.
- Label encode the churn target and one-hot encode model features.
- Standardize numeric features inside the modeling pipeline.
- Use class weighting to address churn class imbalance.

In [ ]:
clean_df = clean_telco_data(raw_df)
model_df = engineer_features(clean_df)
processed_path = save_processed_data(model_df)
model_df.head()

In [ ]:
split = make_train_test_split(model_df)
preprocessor = build_preprocessor(split.numeric_features, split.categorical_features)

print('Training rows:', split.X_train.shape[0])
print('Testing rows:', split.X_test.shape[0])
print('Numeric features:', split.numeric_features)
print('Categorical feature count:', len(split.categorical_features))
print('Label encoding:', dict(zip(split.label_encoder.classes_, split.label_encoder.transform(split.label_encoder.classes_))))

In [ ]:
class_balance = model_df[TARGET_COLUMN].value_counts(normalize=True).rename('share').to_frame()
class_balance['count'] = model_df[TARGET_COLUMN].value_counts()
class_balance

Class imbalance matters because churn customers are the minority class. The project uses class weighting in Logistic Regression, Decision Tree, and Random Forest, plus `scale_pos_weight` for optional XGBoost. SMOTE can also be tested as an alternative, but class weighting keeps the feature pipeline simpler and preserves real customer records.

## 5. Exploratory Data Analysis with Business Interpretation

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(data=model_df, x='Churn', order=['No', 'Yes'], ax=ax)
ax.set_title('Churn Distribution')
ax.set_xlabel('Churn')
ax.set_ylabel('Customers')
plt.show()

model_df['Churn'].value_counts(normalize=True).rename('share').to_frame()

Observation: Churn is typically the minority class, so accuracy alone can be misleading. Business interpretation should focus on how well the model catches actual churners.

In [ ]:
contract_churn = churn_rate_by(model_df, 'Contract')
fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(data=contract_churn, x='Contract', y='churn_rate', ax=ax, color='#cc4125')
ax.set_title('Churn Rate by Contract Type')
ax.set_xlabel('Contract')
ax.set_ylabel('Churn Rate')
plt.show()
contract_churn

Observation: Month-to-month customers are usually the most vulnerable because switching costs are low and there is no long-term commitment. Retention offers should encourage longer contracts without over-discounting low-risk customers.

In [ ]:
payment_churn = churn_rate_by(model_df, 'PaymentMethod')
fig, ax = plt.subplots(figsize=(9, 4))
sns.barplot(data=payment_churn, x='PaymentMethod', y='churn_rate', ax=ax, color='#e69138')
ax.set_title('Churn Rate by Payment Method')
ax.set_xlabel('Payment Method')
ax.set_ylabel('Churn Rate')
ax.tick_params(axis='x', rotation=25)
plt.show()
payment_churn

Observation: Payment method can reveal billing friction. If electronic check customers churn more, the business can test autopay migration campaigns, billing reminders, or payment-experience improvements.

In [ ]:
internet_churn = churn_rate_by(model_df, 'InternetService')
fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(data=internet_churn, x='InternetService', y='churn_rate', ax=ax, color='#3d85c6')
ax.set_title('Churn Rate by Internet Service')
ax.set_xlabel('Internet Service')
ax.set_ylabel('Churn Rate')
plt.show()
internet_churn

Observation: Internet service type helps separate product and pricing risk. Higher-risk internet segments should be reviewed for service quality, price sensitivity, and missing support add-ons.

In [ ]:
tenure_churn = churn_rate_by(model_df, 'tenure_group')
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=tenure_churn, x='tenure_group', y='churn_rate', ax=ax, color='#674ea7')
ax.set_title('Churn Rate by Tenure Group')
ax.set_xlabel('Tenure Group')
ax.set_ylabel('Churn Rate')
ax.tick_params(axis='x', rotation=25)
plt.show()
tenure_churn

Observation: Short-tenure customers are often at higher risk because they have not yet built loyalty or recovered acquisition cost. This supports early-life onboarding and first-year retention programs.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(data=model_df, x='MonthlyCharges', hue='Churn', bins=35, kde=True, common_norm=False, stat='density', ax=ax)
ax.set_title('Monthly Charges Distribution by Churn')
ax.set_xlabel('Monthly Charges')
plt.show()

Observation: Monthly charges can indicate price sensitivity. High monthly charge churners are especially important because they represent more revenue at risk.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.boxplot(data=model_df, x='Churn', y='MonthlyCharges', order=['No', 'Yes'], ax=ax)
ax.set_title('Monthly Charges by Churn Status')
plt.show()

Observation: The boxplot helps compare whether churned customers carry meaningfully different price levels. This is useful for designing discount thresholds and plan-fit outreach.

In [ ]:
senior_churn = churn_rate_by(model_df, 'SeniorCitizen')
fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(data=senior_churn, x='SeniorCitizen', y='churn_rate', ax=ax, color='#76a5af')
ax.set_title('Churn Rate by Senior Citizen Status')
ax.set_xlabel('Senior Citizen')
ax.set_ylabel('Churn Rate')
plt.show()
senior_churn

Observation: Senior customer churn should be handled carefully. If risk is elevated, support accessibility, billing clarity, and proactive assistance may matter more than broad promotions.

In [ ]:
numeric_df = model_df.select_dtypes(include='number').copy()
numeric_df['ChurnFlag'] = model_df['Churn'].eq('Yes').astype(int)
fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(numeric_df.corr(), annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax)
ax.set_title('Correlation Matrix')
plt.show()

Observation: Correlations help identify linear relationships, but churn is driven by interactions across tenure, contract, pricing, and service bundle. This motivates model-based feature importance later.

## 6. Feature Engineering

Engineered features:

- `tenure_group`: captures lifecycle stage and onboarding risk.
- `average_monthly_spend`: approximates customer value and spend intensity.
- `service_count`: captures bundle depth and stickiness.
- `long_term_customer_flag`: separates established customers from newer ones.
- `high_value_customer_flag`: supports revenue-at-risk prioritization.
- `month_to_month_flag`: highlights low-commitment accounts.
- `has_security_or_support`: captures support and protection attachment.
- `auto_payment_flag`: captures payment convenience and billing stability.

In [ ]:
model_df[['tenure', 'tenure_group', 'MonthlyCharges', 'TotalCharges', 'average_monthly_spend', 'service_count', 'long_term_customer_flag', 'high_value_customer_flag']].head()

## 7. Machine Learning Model Development

Models trained:

- Logistic Regression
- Decision Tree
- Random Forest
- XGBoost, if installed

Hyperparameters are tuned with `GridSearchCV` and stratified cross-validation. The final model comparison emphasizes recall and ROC-AUC.

In [ ]:
metrics_df, fitted_models, split = train_and_evaluate_models(model_df, include_xgboost=True)
metrics_df[['model', 'accuracy', 'precision', 'recall', 'f1_score', 'roc_auc', 'false_negatives', 'true_positives']]

## 8. Model Evaluation and Comparison

Recall is especially important because failing to identify a real churner means the business misses the opportunity to intervene. ROC-AUC is also useful because it measures ranking quality across probability thresholds.

In [ ]:
best_model_name, best_model = select_best_model(metrics_df, fitted_models)
print('Best model:', best_model_name)
roc_path = plot_roc_curves(fitted_models, split.X_test, split.y_test)
display(Image(filename=str(roc_path)))

In [ ]:
confusion_path = plot_best_confusion_matrix(best_model, split.X_test, split.y_test)
display(Image(filename=str(confusion_path)))

Business interpretation of the confusion matrix:

- True positives: churn customers correctly identified for retention outreach.
- False negatives: churn customers missed by the model, creating preventable revenue loss.
- False positives: retained customers targeted unnecessarily, creating campaign cost.
- True negatives: retained customers correctly left out of churn campaigns.

## 9. Feature Importance and Explainability

In [ ]:
feature_importance = extract_feature_importance(best_model)
feature_importance.head(20)

In [ ]:
if not feature_importance.empty:
    importance_path = plot_feature_importance(feature_importance, top_n=20)
    display(Image(filename=str(importance_path)))

Interpretation guide:

- Contract type often captures commitment level and switching risk.
- Tenure captures lifecycle maturity and early customer experience.
- Monthly charges capture price sensitivity and revenue exposure.
- Online security and tech support can indicate service value and stickiness.
- Payment method can reveal billing friction or lower commitment behavior.

In [ ]:
# Optional SHAP analysis. This can be slower depending on the selected model.
try:
    import shap
    transformed_train = best_model.named_steps['preprocessor'].transform(split.X_train)
    feature_names = best_model.named_steps['preprocessor'].get_feature_names_out()
    classifier = best_model.named_steps['model']
    explainer = shap.Explainer(classifier, transformed_train, feature_names=feature_names)
    shap_values = explainer(transformed_train[:500])
    shap.plots.beeswarm(shap_values, max_display=20)
except Exception as exc:
    print('SHAP skipped:', exc)

## 10. Save Model Artifacts

In [ ]:
artifacts = save_model_artifacts(metrics_df, fitted_models, split)
artifacts

## 11. Retention Strategy Recommendations

### New customers

- Use onboarding check-ins during the first 90 days.
- Provide first-bill support and explain plan value.
- Monitor early service complaints as churn signals.

### High monthly charge users

- Offer bill reviews and right-plan recommendations.
- Use targeted value-preserving bundles instead of blanket discounts.
- Prioritize high-risk, high-value customers for support callbacks.

### Month-to-month customers

- Offer annual contract incentives and price-lock guarantees.
- Bundle security, tech support, or streaming add-ons to increase stickiness.

### Senior citizens

- Provide clearer billing communication and simplified support.
- Use proactive assistance where technical friction may lead to churn.

### Customers without security or support services

- Offer trial periods for online security and tech support.
- Educate customers on risk protection and service reliability benefits.

### Monitoring system

- Refresh churn scores monthly.
- Track campaign acceptance, save rate, revenue retained, and discount cost.
- Add churn-risk dashboards for customer success and marketing teams.

## 12. Dashboard Extension

Run the Streamlit dashboard from the project root:

```powershell
streamlit run dashboard/streamlit_app.py
```

The Power BI dashboard specification and starter DAX measures are available in the `dashboard/` directory.